# Gene → Tissue Relation Pipeline

Builds a unified, deduplicated edge table for the strictly **Gene–Tissue** relation (excludes Anatomy), integrating aging-specific datasets.

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | kg_type | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**
- BioGrakn\n- Harmonizome\n- iBKH\n- DigitalAgingAtlas (Human & Mouse)

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [17]:
import pandas as pd
import os

BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR     = BASE_DIR + 'processed_data/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_TISSUE/ALL_GENE_TISSUE.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name'
]

## 1 · Mapping DBs

In [18]:
ncbi = pd.read_csv(MAPPING_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_sym2desc_dict = dict(zip(ncbi['Symbol'],  ncbi['description']))
NCBI_syn2sym_dict = {}
for synonyms, symbol in zip(ncbi['Synonyms'], ncbi['Symbol']):
    for syn in str(synonyms).split('|'):
        NCBI_syn2sym_dict[syn.strip()] = symbol
del ncbi

## 2 · Load Source Files

In [19]:
# BioGrakn (Gene -> Tissue)
BioGrakn = pd.read_csv(PROC_DIR + 'BioGrakn/Gene_Tissue_1_Biogrkn.csv')
BioGrakn.columns = BioGrakn.columns.str.lower()
BioGrakn['head_id_is'] = 'NCBI_ID'
BioGrakn['tail_id_is'] = 'BTO_ID'
BioGrakn['relation'] = 'Gene_Tissue'
BioGrakn['head_type'] = 'Gene'
BioGrakn['tail_type'] = 'Tissue'
BioGrakn['kg_source'] = 'BioGrakn'
BioGrakn['kg_type'] = 'Generalised'
print(f"BioGrakn: {len(BioGrakn):,} rows")
display(BioGrakn)

BioGrakn: 84,531 rows


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type
0,TSPAN6,Gene_Tissue,BTO:0000149,Gene,Tissue,BioGrakn,tetraspanin 6,breast,NCBI_ID,BTO_ID,Generalised
1,TSPAN6,Gene_Tissue,BTO:0001340,Gene,Tissue,BioGrakn,tetraspanin 6,bronchus,NCBI_ID,BTO_ID,Generalised
2,TSPAN6,Gene_Tissue,BTO:0000959,Gene,Tissue,BioGrakn,tetraspanin 6,esophagus,NCBI_ID,BTO_ID,Generalised
3,TSPAN6,Gene_Tissue,BTO:0000662,Gene,Tissue,BioGrakn,tetraspanin 6,nasopharynx,NCBI_ID,BTO_ID,Generalised
4,TSPAN6,Gene_Tissue,BTO:0001078,Gene,Tissue,BioGrakn,tetraspanin 6,placenta,NCBI_ID,BTO_ID,Generalised
...,...,...,...,...,...,...,...,...,...,...,...
84526,SCO2,Gene_Tissue,BTO:0000047,Gene,Tissue,BioGrakn,synthesis of cytochrome C oxidase 2,adrenal gland,NCBI_ID,BTO_ID,Generalised
84527,SCO2,Gene_Tissue,BTO:0000365,Gene,Tissue,BioGrakn,synthesis of cytochrome C oxidase 2,duodenum,NCBI_ID,BTO_ID,Generalised
84528,SCO2,Gene_Tissue,BTO:0000651,Gene,Tissue,BioGrakn,synthesis of cytochrome C oxidase 2,small intestine,NCBI_ID,BTO_ID,Generalised
84529,SCO2,Gene_Tissue,BTO:0001363,Gene,Tissue,BioGrakn,synthesis of cytochrome C oxidase 2,testis,NCBI_ID,BTO_ID,Generalised


In [20]:
# Harmonizome (Gene -> Tissue)
Harmonizome = pd.read_csv(PROC_DIR + 'harmonizome/Gene_Tissue_Harmonizome.csv')
Harmonizome.columns = Harmonizome.columns.str.lower()
Harmonizome['head_id_is'] = 'NCBI_ID'
Harmonizome['tail_id_is'] = 'BTO_ID'
Harmonizome['relation'] = 'Gene_Tissue'
Harmonizome['head_type'] = 'Gene'
Harmonizome['tail_type'] = 'Tissue'
Harmonizome['kg_source'] = 'Harmonizome'
Harmonizome['kg_type'] = 'Generalised'
print(f"Harmonizome: {len(Harmonizome):,} rows")
Harmonizome

Harmonizome: 2,188,155 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type
0,LINC00208,Gene_Tissue,BTO:0000562,Gene,Tissue,GTEx,Harmonizome,long intergenic non-protein coding RNA 208,heart,NCBI_ID,BTO_ID,Generalised
1,LIN9,Gene_Tissue,BTO:0000562,Gene,Tissue,GTEx,Harmonizome,lin-9 DREAM MuvB core complex component,heart,NCBI_ID,BTO_ID,Generalised
2,PPFIBP1,Gene_Tissue,BTO:0000562,Gene,Tissue,GTEx,Harmonizome,PPFIA binding protein 1,heart,NCBI_ID,BTO_ID,Generalised
3,REC114,Gene_Tissue,BTO:0000562,Gene,Tissue,GTEx,Harmonizome,REC114 meiotic recombination protein,heart,NCBI_ID,BTO_ID,Generalised
4,VWC2,Gene_Tissue,BTO:0000562,Gene,Tissue,GTEx,Harmonizome,von Willebrand factor C domain containing 2,heart,NCBI_ID,BTO_ID,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...
2188150,SUB1,Gene_Tissue,BTO:0002341,Gene,Tissue,TISSUES Text-mining Tissue Protein Expression ...,Harmonizome,SUB1 regulator of transcription,leaf collar,NCBI_ID,BTO_ID,Generalised
2188151,DNAJC15,Gene_Tissue,BTO:0003504,Gene,Tissue,TISSUES Text-mining Tissue Protein Expression ...,Harmonizome,DnaJ heat shock protein family (Hsp40) member C15,ov-202 cell,NCBI_ID,BTO_ID,Generalised
2188152,STRADA,Gene_Tissue,BTO:0002772,Gene,Tissue,TISSUES Text-mining Tissue Protein Expression ...,Harmonizome,STE20 related adaptor alpha,chela,NCBI_ID,BTO_ID,Generalised
2188153,TNP2,Gene_Tissue,BTO:0002687,Gene,Tissue,TISSUES Text-mining Tissue Protein Expression ...,Harmonizome,transition protein 2,leydig cell tumor cell,NCBI_ID,BTO_ID,Generalised


In [21]:
# iBKH (Gene -> Tissue)
iBKH = pd.read_csv(PROC_DIR + 'iBKH/Gene_Tissue_ibkh.csv')
iBKH.columns = iBKH.columns.str.lower()
iBKH['head_id_is'] = 'NCBI_ID'
iBKH['tail_id_is'] = 'BTO_ID'
iBKH['relation'] = 'Gene_Tissue'
iBKH['head_type'] = 'Gene'
iBKH['tail_type'] = 'Tissue'
iBKH['kg_source'] = 'iBKH'
iBKH['kg_type'] = 'Generalised'
print(f"iBKH: {len(iBKH):,} rows")
iBKH

iBKH: 3,041,850 rows


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,tail_detail_name.1,kg_type
0,A1BG-AS1,Gene_Tissue,BTO:0000608,Gene,Tissue,iBKH,A1BG antisense RNA 1,hepatoma cell,NCBI_ID,BTO_ID,hepatoma cell,Generalised
1,A1BG-AS1,Gene_Tissue,BTO:0000594,Gene,Tissue,iBKH,A1BG antisense RNA 1,liver cancer cell,NCBI_ID,BTO_ID,liver cancer cell,Generalised
2,A1BG-AS1,Gene_Tissue,BTO:0000599,Gene,Tissue,iBKH,A1BG antisense RNA 1,Hep-G2 cell,NCBI_ID,BTO_ID,Hep-G2 cell,Generalised
3,A1BG-AS1,Gene_Tissue,BTO:0000578,Gene,Tissue,iBKH,A1BG antisense RNA 1,hepatoma cell line,NCBI_ID,BTO_ID,hepatoma cell line,Generalised
4,A1BG-AS1,Gene_Tissue,BTO:0006081,Gene,Tissue,iBKH,A1BG antisense RNA 1,hiPSC cell,NCBI_ID,BTO_ID,hiPSC cell,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...
3041845,ZRANB2-DT,Gene_Tissue,BTO:0000930,Gene,Tissue,iBKH,ZRANB2 divergent transcript,neuroblast,NCBI_ID,BTO_ID,neuroblast,Generalised
3041846,ZSCAN16-AS1,Gene_Tissue,BTO:0001251,Gene,Tissue,iBKH,ZSCAN16 antisense RNA 1,liver sinusoidal endothelial cell,NCBI_ID,BTO_ID,liver sinusoidal endothelial cell,Generalised
3041847,ZSCAN16-AS1,Gene_Tissue,BTO:0000876,Gene,Tissue,iBKH,ZSCAN16 antisense RNA 1,monocyte,NCBI_ID,BTO_ID,monocyte,Generalised
3041848,ZSCAN16-AS1,Gene_Tissue,BTO:0002741,Gene,Tissue,iBKH,ZSCAN16 antisense RNA 1,hepatic stellate cell,NCBI_ID,BTO_ID,hepatic stellate cell,Generalised


In [22]:
# Digital Aging Atlas (Human & Mouse) (Gene -> Tissue)
DAA_H = pd.read_csv(PROC_DIR + 'digitalagingatlas/Human/DAA_Gene_Tissue_Human.csv')
DAA_H.columns = DAA_H.columns.str.lower()
DAA_H['kg_source'] = 'DAA_Human'



DAA = DAA_H
DAA['head_id_is'] = 'NCBI_ID'
DAA['tail_id_is'] = 'BTO_ID'
DAA['relation'] = 'Gene_Tissue'
DAA['head_type'] = 'Gene'
DAA['tail_type'] = 'Tissue'
DAA['kg_type'] = 'Aging'
print(f"DigitalAgingAtlas: {len(DAA):,} rows")
DAA

DigitalAgingAtlas: 2,917 rows


,head,relation,tail,head_type,tail_type,relation_source,species,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type
0,ERI1,Gene_Tissue,BTO:0001103,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,exoribonuclease 1,skeletal muscle,NCBI_ID,BTO_ID,DAA_Human,Aging
1,NT5C2,Gene_Tissue,BTO:0001103,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,"5'-nucleotidase, cytosolic II",skeletal muscle,NCBI_ID,BTO_ID,DAA_Human,Aging
2,PTS,Gene_Tissue,BTO:0001103,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,6-pyruvoyltetrahydropterin synthase,skeletal muscle,NCBI_ID,BTO_ID,DAA_Human,Aging
3,ABHD14A,Gene_Tissue,BTO:0000142,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,abhydrolase domain containing 14A,brain,NCBI_ID,BTO_ID,DAA_Human,Aging
4,ABI2,Gene_Tissue,BTO:0000142,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,abl interactor 2,brain,NCBI_ID,BTO_ID,DAA_Human,Aging
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2912,ZNF549,Gene_Tissue,BTO:0003298,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,zinc finger protein 549,mesenchymal stem cell,NCBI_ID,BTO_ID,DAA_Human,Aging
2913,ZNF549,Gene_Tissue,BTO:0000294,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,zinc finger protein 549,dermis,NCBI_ID,BTO_ID,DAA_Human,Aging
2914,ZNF770,Gene_Tissue,BTO:0000975,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,zinc finger protein 770,ovary,NCBI_ID,BTO_ID,DAA_Human,Aging
2915,ZFR,Gene_Tissue,BTO:0000975,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,zinc finger RNA binding protein,ovary,NCBI_ID,BTO_ID,DAA_Human,Aging


## 3 · Consolidate

In [23]:
source_dfs = [BioGrakn, Harmonizome, iBKH, DAA]
aligned = []
for df in source_dfs:
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()]
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])


final_df = pd.concat(aligned, ignore_index=True)
final_df = final_df[final_df['head'].astype(str).str.upper() != 'NAN']

In [24]:
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,TSPAN6,Gene_Tissue,BTO:0000149,Gene,None,Tissue,BioGrakn,Generalised,NCBI_ID,BTO_ID,tetraspanin 6,breast
1,TSPAN6,Gene_Tissue,BTO:0001340,Gene,None,Tissue,BioGrakn,Generalised,NCBI_ID,BTO_ID,tetraspanin 6,bronchus
2,TSPAN6,Gene_Tissue,BTO:0000959,Gene,None,Tissue,BioGrakn,Generalised,NCBI_ID,BTO_ID,tetraspanin 6,esophagus
3,TSPAN6,Gene_Tissue,BTO:0000662,Gene,None,Tissue,BioGrakn,Generalised,NCBI_ID,BTO_ID,tetraspanin 6,nasopharynx
4,TSPAN6,Gene_Tissue,BTO:0001078,Gene,None,Tissue,BioGrakn,Generalised,NCBI_ID,BTO_ID,tetraspanin 6,placenta
...,...,...,...,...,...,...,...,...,...,...,...,...
5317448,ZNF549,Gene_Tissue,BTO:0003298,Gene,None,Tissue,DAA_Human,Aging,NCBI_ID,BTO_ID,zinc finger protein 549,mesenchymal stem cell
5317449,ZNF549,Gene_Tissue,BTO:0000294,Gene,None,Tissue,DAA_Human,Aging,NCBI_ID,BTO_ID,zinc finger protein 549,dermis
5317450,ZNF770,Gene_Tissue,BTO:0000975,Gene,None,Tissue,DAA_Human,Aging,NCBI_ID,BTO_ID,zinc finger protein 770,ovary
5317451,ZFR,Gene_Tissue,BTO:0000975,Gene,None,Tissue,DAA_Human,Aging,NCBI_ID,BTO_ID,zinc finger RNA binding protein,ovary


In [29]:
# Count true NaN values
true_nan_count = final_df.isna().sum()

# Count string 'NAN' values (case-insensitive)
string_nan_count = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

# Combine both counts
nan_summary = pd.DataFrame({
    'NaN_count': true_nan_count,
    "'NAN'_string_count": string_nan_count,
    'Total_NaN_like': true_nan_count + string_nan_count
})
nan_summary

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,5317453,0,5317453
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [28]:
set(final_df['relation']),set(final_df['head_type']),set(final_df['tail_type']),set(final_df['kg_source']),set(final_df['head_id_is']),set(final_df['tail_id_is'])

({'Gene_Tissue'},
 {'Gene'},
 {'Tissue'},
 {'BioGrakn', 'DAA_Human', 'Harmonizome', 'iBKH'},
 {'NCBI_ID'},
 {'BTO_ID'})

## 4 · Gene Name Normalisation

In [25]:
mask = final_df['head_detail_name'].isna()
final_df.loc[mask, 'head'] = final_df.loc[mask, 'head'].astype(str).str.replace('-', '', regex=False).map(NCBI_syn2sym_dict).fillna(final_df.loc[mask, 'head'])
mask = final_df['head_detail_name'].isna()
final_df.loc[mask, 'head_detail_name'] = final_df.loc[mask, 'head'].map(NCBI_sym2desc_dict)

final_df = final_df.dropna(subset=['head_detail_name', 'tail_detail_name']).reset_index(drop=True)
print(f"Consolidated rows: {len(final_df):,}")

Consolidated rows: 5,317,453


## 5 · Deduplication

In [26]:
def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))
group_cols = ['head', 'relation', 'tail']
final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first', 'relation_type': 'first', 'tail_type': 'first',
    'kg_source': merge_sources, 'kg_type': merge_sources, 
    'head_id_is': 'first', 'tail_id_is': 'first',
    'head_detail_name': 'first', 'tail_detail_name': 'first'
})
print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")

Before dedup: 5,317,453  |  After dedup: 4,627,508


In [27]:
# Count true NaN values
true_nan_count = final_df_dedup.isna().sum()

# Count string 'NAN' values (case-insensitive)
string_nan_count = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

# Combine both counts
nan_summary = pd.DataFrame({
    'NaN_count': true_nan_count,
    "'NAN'_string_count": string_nan_count,
    'Total_NaN_like': true_nan_count + string_nan_count
})
nan_summary

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,4627508,0,4627508
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


## 6 · Save Output

In [30]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 4,627,508 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_TISSUE/ALL_GENE_TISSUE.csv
